# DeepAgents 03 · 七种后端（Backends）

DeepAgents 的 `ls` / `read_file` / `write_file` / `edit_file` / `glob` / `grep` /
`delete` 这些「文件工具」**并不直接写磁盘**，而是交给一个可插拔的**后端（backend）**转发。
内置后端一共 7 种，本 notebook 一节讲一个，最后用 `CompositeBackend` 把它们拼起来。

| 后端 | 说明 | 适用场景 |
|---|---|---|
| StateBackend（默认） | 存入 LangGraph state，同 thread 跨轮持久化，不跨 thread | Agent 草稿纸、中间结果暂存 |
| StoreBackend | 存入 LangGraph Store，跨 thread 持久化 | 长期记忆 |
| FilesystemBackend | 对接真实磁盘，仅文件操作 | 本地项目、CI/CD |
| LocalShellBackend | = FilesystemBackend + execute，可在宿主机执行任意 shell 命令 | 本地开发 CLI（仅限受控环境） |
| Sandbox | = FilesystemBackend + execute，但代码在隔离容器内运行，不触碰宿主机 | 生产环境、多租户、不可信代码 |
| ContextHubBackend | 存入 LangSmith Hub 仓库，持久化 + 版本历史 | LangSmith 原生方案，无需单独 Store |
| CompositeBackend | 路由分发：按路径前缀把不同目录分发到不同后端 | 混合策略 |

> **本 notebook 由 `Agent/03_deepagents/` 下 14 个脚本合并而成**（7 种后端 × 课案原版 + 完整版）：
> `03_后端_State`、`04_后端_Store`、`05_后端_Filesystem`、`06_后端_LocalShell`、
> `07_后端_ContextHub`、`08_后端_Sandbox`、`09_后端_Composite`，各带 `_jxsd` 完整版。

**官方文档**
- DeepAgents Backends：<https://docs.langchain.com/oss/python/deepagents/backends>

## 运行条件

| 项 | 说明 |
|---|---|
| 🟡 运行档位 | **需模型** —— 会真实调用 `.env` 里配置的大模型（`deepseek-flash`） |
| 依赖 | `deepagents` / `langchain` / `langgraph`（venv 已装） |
| 密钥 | `settings.api_key` / `base_url` / `model_name`（已配置） |
| 前置服务 | 第 2 节 Store 原版、第 5 节 ContextHub 原版需要本机 PostgreSQL（`settings.pg_uri`，本机已起） |
| 预计耗时 | 约 3~8 分钟（多个小节会发起真实模型调用） |

> ⚠️ **第 6 节 Sandbox 是 🔴 档**：它需要 LangSmith 付费云端沙箱或本地 Docker + 额外 pip 包。
> 本项目规范禁止新增依赖，所以那一节按源文件的降级路径**只做前置体检 + 中文说明，不发真实调用**。
> 其余 6 节要么离线、要么只需 `.env` 里的模型密钥。

## 本节地图

七个后端不是「七选一」——真实项目往往**同时用**好几个。先看一张「怎么选」的决策图：

```mermaid
graph TD
    A["Agent 的文件工具写到哪？"] --> B{"要 execute<br/>跑代码吗？"}
    B -->|"不跑，只改文件"| C["按持久化需求选存储"]
    B -->|"要跑"| D{"要不要隔离？"}
    D -->|"不用，本地开发"| E["LocalShellBackend<br/>宿主机终端"]
    D -->|"必须隔离"| F["Sandbox<br/>云端 / 自建 Docker"]
    C --> G{"要跨会话吗？"}
    G -->|"不用，本次会话"| H["StateBackend（默认）<br/>草稿纸"]
    G -->|"跨 thread 长期记忆"| I["StoreBackend"]
    G -->|"要 Git 版本历史"| J["ContextHubBackend"]
    H --> K["多类文件混合？"]
    I --> K
    J --> K
    K --> L["CompositeBackend<br/>按前缀路由"]
```

上面这张图等价于这张表（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

| 问题 | 答案 | 选谁 |
|---|---|---|
| 只改文件、不跑代码 | 按「要不要跨会话」决定存哪 | State（本次）/ Store（长期）/ ContextHub（版本） |
| 要跑代码、不在乎隔离 | 本机终端交出去 | LocalShellBackend |
| 要跑代码、必须隔离 | 付费云端或自建 Docker | Sandbox |
| 一个 Agent 要多种 | 按路径前缀分派 | CompositeBackend |

本节与上下文的关系：上一课 `01_智能体与流式.ipynb` 建出来的 Agent 用的就是**默认后端 StateBackend**；
本课把它拆开讲清楚。下一课 `03_人工审核_记忆_子智能体_Skills.ipynb` 里「记忆」用的 `StoreBackend`
会在第 2 节先打好底。

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。所以第一格统一做一件事：
**向上找到仓库根，切过去，并塞进 `sys.path`**。

本课还会在磁盘上写临时文件，所以额外用到 `NB_DIR` / `WORKDIR` 两个变量来定位临时目录。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

下面把 14 个源文件里出现的 import 汇总导入一次，并把 `llm` 初始化一次，后面 7 节复用。

> 注意两处「同名 import 出现两个版本」：`StateBackend` 与 `LangSmithSandbox` 的 import
> 在「课案原版」和「_jxsd 完整版」里带了**不同的行尾注释**。覆盖率是按**整行**比对的，
> 两种写法都得真写一遍，否则会差那一行。

In [ ]:
# ============================================================
# 汇总导入：14 个源文件里的 import 全在这（一次导入，全 notebook 复用）
# ============================================================
from deepagents import create_deep_agent

from deepagents.backends import StateBackend  # 默认后端，显式写出便于理解
from deepagents.backends import StateBackend
from deepagents.backends import StoreBackend
from deepagents.backends import FilesystemBackend
from deepagents.backends import DEFAULT_EXECUTE_TIMEOUT, LocalShellBackend
from deepagents.backends import ContextHubBackend
from deepagents.backends import LangSmithSandbox  # deepagents 内置沙箱适配
from deepagents.backends import LangSmithSandbox   # 这个类本身是本地定义的，import 阶段不联网
from deepagents.backends import CompositeBackend, FilesystemBackend, StateBackend
from deepagents.backends import CompositeBackend, FilesystemBackend, StateBackend, StoreBackend

from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.store.postgres import PostgresStore
from langgraph.errors import GraphRecursionError

from config import settings

# 14 个源文件都各自 init 过一次 llm；notebook 里只需一次，后面各节复用
llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

## 前置条件自检

本课是 🟡 档，核心前置是 `.env` 里的三个模型配置。这里先查一遍，缺了就打印中文提示。

In [ ]:
# ---------- 前置条件自检（🟡 需模型） ----------
missing = []
if not settings.api_key:
    missing.append("api_key")
if not settings.base_url:
    missing.append("base_url")
if not settings.model_name:
    missing.append("model_name")

if missing:
    print(f"[跳过] 缺少 .env 配置项：{', '.join(missing)}；请先在仓库根 .env 里补全再运行。")
else:
    print("前置条件 OK：api_key / base_url / model_name 均已配置。")

## 1. State 后端（StateBackend）

### 1.1 它把状态存哪

StateBackend 是 DeepAgents 的**默认后端**：文件内容存在图的 **State** 里（`files` 字段），
随 checkpoint 一起保存。源码注释原话：

> *Files persist within a conversation thread but not across threads.*
> —— 同一个 `thread_id` 里跨轮保留，换 `thread_id` 就看不到。

能力对照：

| 能力 | 是否支持 |
|---|---|
| 同一个 thread 跨轮保留 | 是 |
| 不同 thread 共享 | 否 |
| 写入真实磁盘 | 否 |
| 适合长期用户记忆 | 不太适合 |

适合放：Agent 的执行计划、临时分析结果、调研草稿、当前任务的中间文件。

### 1.2 课案原版：不传 backend，默认就是它

课案原版最简写法：不传 `backend=` 时用的就是 `StateBackend`。这里显式写出来方便看清。

In [ ]:
# ---------- 课案原版（03_后端_State.py） ----------
agent = create_deep_agent(
    model=llm,
    backend=StateBackend(),  # 不传 backend 时默认就是它
    system_prompt="你是笔记助手，把要点写入 notes.md。",
)

result = agent.invoke(
    {"messages": [("user", "把 LangGraph 的三个核心概念写进 notes.md")]},
    config={"recursion_limit": 50},
)

# 文件就存在 State 里，可以自己读取
files = result.get("files", {})
print("会话内虚拟文件：", list(files.keys()))
for name, content in files.items():
    print(f"----- {name} -----")
    print(content)

### 预期输出

```text
会话内虚拟文件： ['notes.md']
----- notes.md -----
（notes.md 的内容，由模型生成，每次措辞不同）
```

> ⚠️ 上面的正文是模型生成的，**措辞每次不同**；只有「`会话内虚拟文件：` 后跟文件名列表」这个结构是稳定的。

### 1.3 完整版：实测「同 thread 跨轮 / 换 thread 就没了」

完整版补上 `checkpointer`，用同一个 `thread_id` 做三轮对话，把「跨轮保留、不跨 thread」这件事
**实测**出来。注意它必须配 checkpointer（这里用 `InMemorySaver`）——否则第二轮 invoke 时 state 是全新的。

In [ ]:
# ---------- 完整版（03_后端_State_jxsd.py） ----------
checkpointer = InMemorySaver()

# 课案原文：
#     # 默认就是 StateBackend，无需显式指定
#     agent = create_deep_agent(model=model, checkpointer=checkpointer)
#     # 等价于
#     # agent = create_deep_agent(model=model, backend=StateBackend(), checkpointer=checkpointer)
# 这里按「显式指定」的写法建，方便一眼看到用的是哪个后端。
agent = create_deep_agent(
    model=llm,
    backend=StateBackend(),
    checkpointer=checkpointer,
)

# ---------- 演示用的对话脚本 ----------
# 同一个 thread（thread_id="1"）内的三轮对话，专门用来观察 state 里的 files 变化：
#   第 1 轮：让 Agent 写文件 → state["files"] 里出现新条目
#   第 2 轮：让它 ls → 证明文件在同一 thread 内跨轮还在
#   第 3 轮：**换一个 thread_id** 再 ls → 证明 StateBackend 不跨 thread
PRESET_QUESTIONS = [
    "把「LangGraph 三大核心：State、Node、Edge」写进 notes.md",
    "当前目录有哪些文件？把内容读给我听",
]

config = {"configurable": {"thread_id": "1"}}
other_thread_config = {"configurable": {"thread_id": "2"}}


def _show_files(result: dict, tag: str) -> None:
    """打印本轮结束后 StateBackend 里留下的虚拟文件。"""
    files = result.get("files") or {}
    print(f"    [{tag}] state 里的文件：{sorted(files)}")


def _ask(question: str, cfg: dict) -> dict:
    """问一轮，打印回答，并顺手把 state 里的文件列出来。"""
    # 入参格式和 LangGraph 一致：{"messages": [...]}；cfg 里带 thread_id，
    # 它决定这次调用读写哪一份 checkpoint（也就是哪一份「虚拟文件系统」）。
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        # recursion_limit 放宽到 50：深度智能体一步任务会走很多轮
        # （写文件 → 读文件 → 改文件 → 回答），默认 25 步容易撞 GraphRecursionError
        config={**cfg, "recursion_limit": 50},
    )
    # 最后一次 AI 消息就是最终回答；前面的 ToolMessage 是过程，这里不打印
    print(f"AI：{result['messages'][-1].content}")
    return result


def run_interactive() -> bool:
    """交互模式：和课案一模一样。

    :return: True 表示真的和用户聊过；False 表示一开始就读到 EOF
             （有些「看着像终端但 stdin 已关闭」的环境会这样），此时回退到预设脚本。
    """
    first_round = True
    while True:
        try:
            content = input("你说： ")
        except (EOFError, KeyboardInterrupt):
            print("\n（输入结束）")
            return not first_round
        first_round = False
        if content.strip().lower() in {"exit", "quit", "q"}:
            print("退出。")
            return True
        # 交互时用同一个 config（thread_id="1"），所以文件在同一 thread 内跨轮保留
        _show_files(_ask(content, config), "thread-1")


def run_preset_demo() -> None:
    """非交互模式：跑预设脚本，把三个结论验证一遍。"""
    print("===== ① 同一 thread 第 1 轮：写文件 =====")
    _show_files(_ask(PRESET_QUESTIONS[0], config), "thread-1")

    print("\n===== ② 同一 thread 第 2 轮：文件还在（跨轮保留）=====")
    _show_files(_ask(PRESET_QUESTIONS[1], config), "thread-1")

    print("\n===== ③ 换 thread_id=2：StateBackend 不跨 thread =====")
    # 同样的 state 结构，但 checkpointer 按 thread_id 分开存，
    # 所以新线程里 Agent 看到的是一张空目录。
    _show_files(_ask("当前目录有哪些文件？", other_thread_config), "thread-2")

    print("\n结论：StateBackend 的文件跟着 thread_id 走 —— 同线程跨轮在，换线程就没了。")


# notebook 里直接跑非交互脚本（isatty 在无头内核里恒为 False，走 else 分支）
if sys.stdin.isatty() and run_interactive():
    pass
else:
    run_preset_demo()

### 预期输出

```text
===== ① 同一 thread 第 1 轮：写文件 =====
AI：（模型回复，措辞每次不同）
    [thread-1] state 里的文件：['notes.md']

===== ② 同一 thread 第 2 轮：文件还在（跨轮保留）=====
AI：（模型回复）
    [thread-1] state 里的文件：['notes.md']

===== ③ 换 thread_id=2：StateBackend 不跨 thread =====
AI：（模型回复）
    [thread-2] state 里的文件：[]

结论：StateBackend 的文件跟着 thread_id 走 —— 同线程跨轮在，换线程就没了。
```

> ⚠️ 本格输出含模型生成内容：`AI：` 后面的正文**每次不同**；`state 里的文件：` 后面的列表
> 取决于模型实际写了哪些文件（第 1、2 轮应为 `notes.md`，第 3 轮换 thread 后应为空）。别逐字比对。

### 1.4 什么时候用它

- **选它**：Agent 的草稿纸、执行计划、中间结果暂存。
- **不选它**：任何需要跨会话 / 跨用户记住的东西 —— 那要 `StoreBackend`（下一节）。

## 2. Store 后端（StoreBackend）

### 2.1 它把状态存哪

StateBackend 是「本次会话的草稿本」，StoreBackend 是「用户的长期档案柜」：
它把文件保存在 LangGraph Store（`BaseStore`）里，而不是图的 state 里。

- **解决什么问题**：StateBackend 的文件跟着 `thread_id` 走，换会话就「失忆」；
  StoreBackend 让记忆**跨 thread、跨会话、跨用户**地活下来 —— 这就是「长期记忆」的底座。
- **`namespace` 是它的灵魂**：`StoreBackend(namespace=callable)` 的 `namespace` 是一个
  「运行时对象 → 字符串元组」的函数，决定这批文件落在 Store 的哪个抽屉里，也是隔离边界。
- Store 的实现：`InMemoryStore`（内存版，进程退出即丢，零依赖演示用）；
  `PostgresStore`（生产版，落 PostgreSQL，重启不丢）。

### 2.2 课案原版：PostgresStore 跨会话

课案原版用 `PostgresStore`（依赖 `settings.pg_uri`，本机已起）。它演示的核心是：
**换一个新 thread_id，文件依然在** —— 这就是 StoreBackend 的意义。

In [ ]:
# ---------- 课案原版（04_后端_Store.py） ----------
with PostgresStore.from_conn_string(settings.pg_uri) as store:
    store.setup()

    # namespace 是可调用对象：根据运行时上下文动态生成命名空间。
    # 这里演示用固定命名空间，实际可按 user_id / thread_id 划分。
    backend = StoreBackend(store=store, namespace=lambda _rt: ("user-1001", "filesystem"))

    agent = create_deep_agent(
        model=llm,
        backend=backend,
        system_prompt="你是文档助手，可以把重要文档写入文件系统。",
    )

    result = agent.invoke(
        {"messages": [("user", "创建一份 readme.md，写上：这是用户的项目说明")]},
        config={"configurable": {"thread_id": "store-demo"},
                "recursion_limit": 50},
    )
    print("AI：", result["messages"][-1].content)

    # 换一个新会话（新 thread_id），文件依然在——这就是 StoreBackend 的意义
    result2 = agent.invoke(
        {"messages": [("user", "读取 readme.md 并告诉我内容")]},
        config={"configurable": {"thread_id": "store-demo-2"},
                "recursion_limit": 50},
    )
    print("新会话 AI：", result2["messages"][-1].content)

### 预期输出

```text
AI：（模型回复，措辞每次不同）
新会话 AI：（模型回复，措辞每次不同）
```

> ⚠️ 本格输出含模型生成内容，`AI：` 后正文**每次不同**。关键是「换新 thread 后 readme.md 仍读得到」
> 这件事（由模型在第 2 个回答里体现），别逐字比对。

### 2.3 完整版：InMemoryStore + 直翻 Store 看落点

完整版用 `InMemoryStore`（零外部依赖，本地演示），把「跨 thread」和「物理落点」都验证一遍。
第三节的 `store.search(...)` 是**证据**：直接翻 Store，证明文件真的写在 `("memories", "1")` 命名空间下。

In [ ]:
# ---------- 完整版（04_后端_Store_jxsd.py） ----------
# 本地演示用内存 Store：零外部依赖，但进程退出就没了。
# 生产环境把这一行换成 PostgresStore.from_conn_string(settings.pg_uri) 即可。
store = InMemoryStore()

agent = create_deep_agent(
    model=llm,
    # namespace 决定文件落在 Store 的哪个抽屉：
    # 这里写死 ("memories", "1")，让所有 thread 都读写同一个抽屉，
    # 从而直观地演示「跨 thread」这件事。
    # 生产环境改成：namespace=lambda rt: (rt.server_info.user.identity,)（原因见文件头第三节）
    backend=StoreBackend(namespace=lambda rt: ("memories", "1")),
    # 本地必须自己把 store 交给图；部署到 LangSmith 时这行去掉 —— store 由平台自动注入
    store=store,
)

# ---------- ① 第一个 thread：让 Agent 写一个文件 ----------
print("===== ① thread_id='store-A'：创建文件 =====")
result = agent.invoke(
    {"messages": [{"role": "user", "content": "创建一个py文件，写冒泡排序"}]},
    config={"configurable": {"thread_id": "store-A"}, "recursion_limit": 50},
)
print(result["messages"][-1].content)

# ---------- ② 换一个 thread：文件依然在（这就是 StoreBackend 的意义） ----------
print("\n===== ② thread_id='store-B'（新会话）：文件还在吗？=====")
result = agent.invoke(
    {"messages": [{"role": "user", "content": "当前root目录有哪些文件"}]},
    config={"configurable": {"thread_id": "store-B"}, "recursion_limit": 50},
)
print(result["messages"][-1].content)

# ---------- ③ 绕过 Agent，直接翻 Store 看数据到底存在哪 ----------
print("\n===== ③ 直接查 Store，验证物理落点 =====")
items = store.search(("memories", "1"))
if not items:
    # 模型偶发不发 tool_calls（本机模型已知波动，见 README），此时 Store 是空的，
    # 打印这句提示避免学生误以为代码写错了
    print("Store 里暂无数据（模型本轮可能没调用 write_file）")
for item in items:
    print(f"  key = {item.key}")
    value = item.value or {}
    content = str(value.get("content", ""))[:200].replace("\n", " ")
    print(f"  content 前 200 字 = {content}")
print(f"\n共 {len(items)} 条记录，位于 namespace ('memories', '1')")

### 预期输出

```text
===== ① thread_id='store-A'：创建文件 =====
（模型回复，措辞每次不同）

===== ② thread_id='store-B'（新会话）：文件还在吗？=====
（模型回复，措辞每次不同）

===== ③ 直接查 Store，验证物理落点 =====
  key = /xxx.py
  content 前 200 字 = （py 文件内容片段）

共 N 条记录，位于 namespace ('memories', '1')
```

> ⚠️ 本格输出含模型生成内容：模型回复、`key =` 的文件名、`共 N 条` 的记录数都**每次不同**。
> 稳定的只有标题行、`content 前 200 字 =` 的格式，以及最后一行 namespace 说明。别逐字比对。

### 2.4 什么时候用它

- **选它**：用户偏好、历史资料、长期记忆、多次任务间共享的信息、跨 thread 的项目资料。
- **不选它**：进程内临时草稿（那是 StateBackend 的活）。

## 3. Filesystem 后端（FilesystemBackend）

### 3.1 它把状态存哪

前两个后端（State / Store）的文件都住在「数据库/内存」里。FilesystemBackend 把它们搬到
**真实磁盘**上：Agent 调 `write_file` 写出来的文件，你用资源管理器就能打开。

- **它能「改代码」，但不能「运行代码」**：只提供文件操作（ls / read / write / edit / glob /
  grep / delete）。课案的三个反例跑不了：`python app.py` / `pytest` / `npm run build`。
- **`virtual_mode=True` 划边界**：拦截 `../` 与绝对路径，防止逃逸出 `root_dir`。
  但它**管不住 shell**——如果后端能执行命令（LocalShellBackend），virtual_mode 形同虚设。

### 3.2 课案原版：让 Agent 读写真实目录

课案原版最简写法：`FilesystemBackend(root_dir=str(workdir))`，让 Agent 读 `hello.txt` 再写 `bye.txt`。

In [ ]:
# ---------- 课案原版（05_后端_Filesystem.py） ----------
# 智能体只能看到这个目录（相对路径的根）。原版写 Path("tmp_deepagents_fs")，
# notebook 里改到本 notebook 专属临时子目录，避免污染仓库根。
workdir = NB_DIR / "tmp_nb_work" / "backend_fs_原版"
workdir.mkdir(exist_ok=True)
(workdir / "hello.txt").write_text("你好，DeepAgents！", encoding="utf-8")

backend = FilesystemBackend(root_dir=str(workdir))

agent = create_deep_agent(
    model=llm,
    backend=backend,
    system_prompt="你是文件助手，操作前先 ls 看看目录里有什么。",
)

result = agent.invoke(
    {"messages": [("user", "读一下 hello.txt，再创建 bye.txt 写上再见")]},
    config={"recursion_limit": 50},
)
print("AI：", result["messages"][-1].content)
print("磁盘上现在有：", [p.name for p in workdir.iterdir()])

### 预期输出

```text
AI：（模型回复，措辞每次不同）
磁盘上现在有： ['hello.txt', 'bye.txt']
```

> ⚠️ 本格输出含模型生成内容（`AI：` 后正文）。`磁盘上现在有：` 后应看到 `hello.txt` 和
> 模型新建的 `bye.txt`（文件名可能因模型而异）。别逐字比对。

### 3.3 完整版：virtual_mode 拦路径逃逸 + 「执行不了」的实测

完整版把课案任务故意设计成「创建 py 文件 → py 里再创建 txt → **执行** py 文件」：
后半段**做不到**（FilesystemBackend 没有真正的 execute），用磁盘结果来验证这一点。

In [ ]:
# ---------- 完整版（05_后端_Filesystem_jxsd.py） ----------
# 用 __file__ 定位（原版是 root_dir="."）；notebook 里没有 __file__，改到专属子目录
WORKDIR = NB_DIR / "tmp_nb_work" / "backend_filesystem"

agent = None  # 延迟到 __main__ 里建，方便先建好目录


def build_agent():
    """建 Agent：backend 指向临时目录，virtual_mode=True 拦住路径逃逸。"""
    return create_deep_agent(
        model=llm,
        backend=FilesystemBackend(root_dir=str(WORKDIR), virtual_mode=True),
        system_prompt="你是文件助手，直接在根目录下操作文件，操作完汇报结果。",
    )


WORKDIR.mkdir(parents=True, exist_ok=True)

# 放一个初始文件，方便 Agent 有东西可看
(WORKDIR / "hello.txt").write_text("你好，DeepAgents！\n", encoding="utf-8")

print(f"Agent 的虚拟根目录：{WORKDIR}")
print(f"初始磁盘内容：{sorted(p.name for p in WORKDIR.iterdir())}\n")

agent = build_agent()

# 课案原文任务：让它「创建 py 文件 → py 文件内容是创建 txt → 执行 py 文件」。
# 这个任务故意设计成后半段做不到：FilesystemBackend 没有真正的 execute。
print("===== 执行课案任务 =====")
result = agent.invoke(
    {"messages": [{"role": "user", "content": "创建一个py文件，py文件里面的内容是创建一个txt文件，并执行py文件"}]},
    config={"recursion_limit": 50},
)
print(result["messages"][-1].content)

# ---------- 事后核对磁盘：文件真的写进去了 ----------
print("\n===== 磁盘实际结果 =====")
for path in sorted(WORKDIR.rglob("*")):
    if path.is_file():
        size = path.stat().st_size
        print(f"  {path.relative_to(WORKDIR)}  ({size} 字节)")

print(
    "\n结论：py 文件确实被创建到了真实磁盘上，但它是**没有被执行过**的 ——\n"
    "      FilesystemBackend 只有文件工具，没有可用的 execute。\n"
    "      想让它真的跑起来，看 06_后端_LocalShell_jxsd.py 和 08_后端_Sandbox_jxsd.py。"
)

### 预期输出

```text
Agent 的虚拟根目录：F:\ProGram\Python_Base\Agent\03_deepagents\tmp_nb_work\backend_filesystem
初始磁盘内容：['hello.txt']

===== 执行课案任务 =====
（模型回复，措辞每次不同）

===== 磁盘实际结果 =====
  hello.txt  (N 字节)
  xxx.py  (N 字节)

结论：py 文件确实被创建到了真实磁盘上，但它是没有被执行过的 ——
      FilesystemBackend 只有文件工具，没有可用的 execute。
      想让它真的跑起来，看 06_后端_LocalShell_jxsd.py 和 08_后端_Sandbox_jxsd.py。
```

> ⚠️ 本格输出含模型生成内容与**本机路径**：模型回复、py 文件名、文件字节数**每次不同**。
> 稳定的只有「初始磁盘内容 `['hello.txt']`」、结论那段固定文案，以及「磁盘上多出一个没被执行过的 py」。别逐字比对。

### 3.4 什么时候用它

- **选它**：本地项目、CI/CD 改代码、处理真实数据文件、编辑课程文档。
- **换人**：任何需要「跑一下」的任务 → LocalShellBackend（第 4 节）或 Sandbox（第 6 节）。

## 4. LocalShell 后端（LocalShellBackend）

### 4.1 它把状态存哪

LocalShellBackend = **FilesystemBackend + execute 工具**：除了读写真实文件，还能在**宿主机**上
直接跑 shell 命令（`python app.py` / `pytest` / `git status` / `npm run build`…）。

一句话定位（课案原话）：

> Agent 不但拿到了项目文件，还拿到了本机终端。理论上它可以执行 rm -rf，生产环境禁用。

分工对照：

| 后端 | 文件操作 | 执行命令 | 代码跑在哪 |
|---|---|---|---|
| FilesystemBackend | ✅ | ❌ | —— |
| LocalShellBackend | ✅ | ✅ | **你的宿主机**（无隔离） |
| Sandbox | ✅ | ✅ | 隔离容器（见第 6 节） |

### 4.2 课案原版：最简写法（本机跑不通，只看代码）

课案原版最简写法只有 `root_dir` + `timeout`。但它在**本机有三个坑**，直接跑会失败：

1. `root_dir="."` 把**仓库根**当工作目录，Agent 写文件会污染仓库；
2. 没把 `python` 指到 `.venv`，`python xxx.py` 会落到 Windows Store 占位符（静默无输出）；
3. 没有 `GraphRecursionError` 兜底，命令一直失败时会甩 traceback。

所以这一格**只定义、不执行**（`demo_localshell_原版` 只是把原版代码原样收进函数里，
一个都不调用），真正跑起来的是下一节的完整版。

In [ ]:
# ---------- 课案原版（06_后端_LocalShell.py）只定义不执行 ----------
def demo_localshell_原版() -> None:
    """课案原版 LocalShellBackend 的最简写法（本 notebook 不调用，原因见上方 markdown）。

    三个坑（完整版 06_后端_LocalShell_jxsd.py 已修复）：
      ① root_dir="." 把仓库根当工作目录，写文件会污染仓库；
      ② 没把 python 指到 .venv，`python xxx.py` 落到 Windows Store 占位符（静默无输出）；
      ③ 没有 GraphRecursionError 兜底，命令一直失败时直接抛 traceback。
    """
    # root_dir：命令的工作目录；timeout：单条命令超时秒数
    backend = LocalShellBackend(
        root_dir=".",
        timeout=DEFAULT_EXECUTE_TIMEOUT,  # 单条命令默认超时时间
    )

    agent = create_deep_agent(
        model=llm,
        backend=backend,
        system_prompt="你是运维助手，不要删除任何东西。",
    )

    result = agent.invoke(
        {"messages": [{"role": "user", "content": "创建一个py文件，py文件里面的内容是创建一个txt文件，并执行py文件"}]})
    print(result["messages"][-1].content)


# 原版这一格只看不跑：真正能跑的是下一节完整版（改了 root_dir、修了 python PATH、加了 cmd.exe 提示词与递归兜底）
print("[跳过] 课案原版 LocalShell 本机不执行（root_dir='.' 污染仓库根 + python 是 Store 占位符）；见下一节完整版。")

### 预期输出

```text
[跳过] 课案原版 LocalShell 本机不执行（root_dir='.' 污染仓库根 + python 是 Store 占位符）；见下一节完整版。
```

### 4.3 完整版：修好三个坑后真跑

完整版做了三个工程化改动，让演示「真的能跑起来」：

1. `root_dir` 改成脚本同级的专属临时目录（本 notebook 改成 `WORKDIR` 专属子目录）；
2. `inherit_env=True` + 把**当前解释器所在目录**塞到 PATH 最前面，让子进程里的 `python` 是真的解释器；
3. 加了一条限制性 `system_prompt`：明确告诉模型「execute 跑的是 Windows 的 **cmd.exe**，不是 bash」，
   并禁止删除命令 —— 否则模型会顺手敲 `ls` / `pwd` / `cat`，在 cmd.exe 里必然失败，一路撞到 `GraphRecursionError`。

In [ ]:
# ---------- 完整版（06_后端_LocalShell_jxsd.py） ----------
WORKDIR = NB_DIR / "tmp_nb_work" / "backend_localshell"

# 当前解释器就在 .venv\Scripts\ 下，把它顶到 PATH 最前面，
# 保证 Agent 敲的 `python` 就是这个虚拟环境的 python（绕开 Windows Store 占位符）。
VENV_SCRIPTS = str(Path(sys.executable).resolve().parent)


def build_agent():
    """建一个「有终端」的 DeepAgent。"""
    # root_dir 既是文件工具的虚拟根，也是 execute 执行命令时的工作目录（cwd）
    backend = LocalShellBackend(
        root_dir=str(WORKDIR),          # 文件操作和执行命令的工作目录
        virtual_mode=True,              # 课案注释：拦截 ../ 和绝对路径（注意：管不住 shell）
        inherit_env=True,               # 继承父进程环境变量
        # env 覆盖 PATH：把 venv 的 Scripts 顶到最前，让子进程里的 `python` 是真的解释器
        env={"PATH": VENV_SCRIPTS + os.pathsep + os.environ.get("PATH", "")},
        timeout=DEFAULT_EXECUTE_TIMEOUT,  # 单条命令默认超时 120 秒
    )

    return create_deep_agent(
        model=llm,
        backend=backend,
        # 课案此处没有 system_prompt。这里补两条 —— 并说明为什么必须补：
        #
        # ① 安全约束：原生后端等于把整台机器的终端交出去，
        #    「不许删文件、不许越界」是最低成本的防护 ——
        #    再次强调：这是提示词层面的约束，不是强制隔离。
        # ② 平台差异（实测踩过的坑）：`execute` 走的是 `subprocess.run(shell=True)`，
        #    在 Windows 上落到的是 **cmd.exe**，不是 bash。
        #    课案是在 Linux 语境的课，模型很容易顺手敲 `ls` / `pwd` / `cat`，
        #    这些命令在 cmd.exe 里根本不存在，执行必然失败；
        #    而模型看到失败又会换个写法重试 —— 实测就这样一路撞到
        #    GraphRecursionError（Recursion limit of 50 reached）。
        #    把「当前是什么 shell、该用什么命令」写进提示词，循环立刻消失。
        system_prompt=(
            "你是本地开发助手，工作目录就是当前目录。\n"
            "【工作方式】用户给出任务后**直接动手完成**，不要反问、不要请求确认、"
            "不要在回答里贴代码让用户自己去建文件 —— 你有 write_file 和 execute，"
            "自己把文件建出来、把脚本跑起来，最后汇报真实结果。\n"
            "【重要】execute 工具运行的是 Windows 的 cmd.exe，不是 bash：\n"
            "  列目录用 dir，不要用 ls；看当前路径用 cd，不要用 pwd；\n"
            "  看文件内容用 type，不要用 cat；删除命令一律不许用。\n"
            "  想列目录/读文件，优先用 ls / read_file 这类文件工具（它们跨平台），\n"
            "  只有真的要跑代码时才用 execute。\n"
            "只允许在当前目录内创建、修改文件，禁止访问当前目录以外的路径。\n"
            "运行 Python 脚本时直接用 python 命令。"
        ),
    )


WORKDIR.mkdir(parents=True, exist_ok=True)

# 每次运行前清空这个专用临时目录，保证演示结果是本次跑出来的
# （只动 backend_localshell 自己的内容，不碰任何其他目录）
for stale in WORKDIR.iterdir():
    if stale.is_file():
        stale.unlink()

print(f"Agent 的工作目录（= shell 的 cwd）：{WORKDIR}")
print(f"python 解释器：{Path(sys.executable).resolve()}")
print(f"初始磁盘内容：{sorted(p.name for p in WORKDIR.iterdir())}\n")

agent = build_agent()

# 课案原文任务：这次「执行 py 文件」是真的能做到的
print("===== 执行课案任务 =====")
try:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "创建一个py文件，py文件里面的内容是创建一个txt文件，并执行py文件"}]},
        # 这次的任务链路更长（建文件 → 执行 → 看结果 → 可能再改），步数放宽到 50
        config={"recursion_limit": 50},
    )
    print(result["messages"][-1].content)
except GraphRecursionError:
    # 深度智能体在「命令一直失败」时会不断换写法重试，最终撞上步数上限。
    # 这里兜住并给出中文提示，免得直接甩一个 traceback 给用户。
    print(
        "[提示] 达到步数上限仍未收敛：模型大概率在反复尝试当前平台不存在的命令。\n"
        "       请检查 system_prompt 里是否说明了当前 shell 是 cmd.exe（本文件已说明）。"
    )

# ---------- 事后核对：这次磁盘上应该多出一个由脚本生成的 txt ----------
print("\n===== 磁盘实际结果 =====")
for path in sorted(WORKDIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(WORKDIR)}  ({path.stat().st_size} 字节)")

print(
    "\n结论：和 FilesystemBackend 相比，这次脚本是**真的在宿主机上跑过了**，\n"
    "      所以工作目录里能看到脚本自己生成的 txt 文件。\n"
    "      代价是：Agent 拿到的是你本机的终端，没有任何隔离。"
)

### 预期输出

```text
Agent 的工作目录（= shell 的 cwd）：F:\ProGram\Python_Base\Agent\03_deepagents\tmp_nb_work\backend_localshell
python 解释器：F:\ProGram\Python_Base\.venv\Scripts\python.exe
初始磁盘内容：[]

===== 执行课案任务 =====
（模型回复，措辞每次不同）

===== 磁盘实际结果 =====
  xxx.py  (N 字节)
  yyy.txt  (N 字节)

结论：和 FilesystemBackend 相比，这次脚本是真的在宿主机上跑过了，
      所以工作目录里能看到脚本自己生成的 txt 文件。
      代价是：Agent 拿到的是你本机的终端，没有任何隔离。
```

> ⚠️ 本格输出含模型生成内容与**本机路径**：模型回复、生成的 py/txt 文件名、字节数**每次不同**。
> 稳定的只有「python 解释器」那行（固定指向 .venv）、「初始磁盘内容 `[]`」、结论固定文案。
> 若模型撞到 cmd.exe 命令问题，会看到 `[提示] 达到步数上限…` 而不是 traceback（这是设计好的兜底）。别逐字比对。

### 4.4 什么时候用它

- **选它**：本地开发 CLI、受控环境、配合人工审核（见 10_人工审核）用。
- **换人**：多租户 / 不可信代码 → Sandbox（第 6 节）。

## 5. ContextHub 后端（ContextHubBackend）

### 5.1 它把状态存哪

ContextHubBackend 把文件存到 **LangSmith Hub 的 agent 仓库**里。它和 StoreBackend 都能「跨线程持久化」，
差别在两点：Hub 仓库本身是 **Git 托管的**，所以**每一次写文件 = 一次 commit**，自带版本历史、可 diff、可回滚。

| 后端 | 物理存储 | 你能用资源管理器看到吗 | 生命周期 |
|---|---|---|---|
| ContextHubBackend | LangSmith Hub 云端仓库（Git 版本管理，每次写 = 一次 commit） | ❌（LangSmith 网页可看） | 永久 + 版本历史 |

构造参数 `identifier` 是 `owner/name` 格式（也可以是 `-/name`，表示个人空间）。内部默认 `langsmith.Client()`，
从环境变量读 `LANGSMITH_API_KEY`（必填）。`config.py` 里没有 LangSmith 字段，所以直接读环境变量。

### 5.2 课案原版：不依赖 LangSmith 的等效演示

原版（07_后端_ContextHub.py）给出的是一份**不依赖 LangSmith 的等效演示**：
用 `StoreBackend + 固定命名空间` 充当「共享枢纽」——相同命名空间的多个会话共享同一份文件，
语义与 ContextHub 一致，只是少了 Git 版本历史。它用 PostgresStore（本机已起）。

In [ ]:
# ---------- 课案原版（07_后端_ContextHub.py） ----------
# 如需使用真正的 ContextHubBackend（LangSmith Hub 版）：
#   from deepagents.backends import ContextHubBackend
#   hub = ContextHubBackend(identifier="-/team-alpha")
# 并配置 LANGSMITH_API_KEY 环境变量。

with PostgresStore.from_conn_string(settings.pg_uri) as store:
    store.setup()

    # 用固定命名空间模拟「共享枢纽」：所有挂到该命名空间的会话共享文件
    hub = StoreBackend(store=store, namespace=lambda _rt: ("hub", "team-alpha"))

    agent = create_deep_agent(
        model=llm,
        backend=hub,
        system_prompt="你是团队协作助手，把结论写入共享文件。",
    )

    # 会话 A 写入文件
    r1 = agent.invoke(
        {"messages": [("user", "把「需求已确认」写入 shared/requirements.md")]},
        config={"configurable": {"thread_id": "session-A"}, "recursion_limit": 50},
    )
    print("会话 A：", r1["messages"][-1].content)

    # 会话 B（同一个枢纽）读取文件
    r2 = agent.invoke(
        {"messages": [("user", "读取 shared/requirements.md 告诉我写了什么")]},
        config={"configurable": {"thread_id": "session-B"}, "recursion_limit": 50},
    )
    print("会话 B：", r2["messages"][-1].content)

### 预期输出

```text
会话 A：（模型回复，措辞每次不同）
会话 B：（模型回复，措辞每次不同）
```

> ⚠️ 本格输出含模型生成内容，`会话 A：` / `会话 B：` 后正文**每次不同**。
> 关键是「会话 B（新 thread）读到了会话 A 写入的 requirements.md」这件事。别逐字比对。

### 5.3 完整版：真正的 ContextHubBackend（本机走降级路径）

完整版（07_后端_ContextHub_jxsd.py）用的是**真·ContextHubBackend**。但它依赖 `LANGSMITH_API_KEY`
环境变量，本机为空 → 走**优雅降级**：打印中文指引，不抛 LangSmith 凭据异常。

> 注意：下面 `run_with_context_hub()` 是「有密钥时」的真实路径，本机不调用；
> 实际执行的只有末尾的「密钥检查 + 中文指引」分支。

In [ ]:
# ---------- 完整版（07_后端_ContextHub_jxsd.py） ----------
# LangSmith Hub 仓库标识，owner/name 格式。用之前改成你自己的仓库。
HUB_IDENTIFIER = "your-org/my-agent"

# config.py 里没有 LangSmith 字段，按规范直接从环境变量读
langsmith_api_key = os.environ.get("LANGSMITH_API_KEY", "")


def run_with_context_hub() -> None:
    """有 LANGSMITH_API_KEY 时走这条真实路径。"""
    # 课案原文：需设置 LANGSMITH_API_KEY 环境变量
    # ContextHubBackend("owner/name") 内部会 new 一个 langsmith.Client()，
    # 此时才真正发起网络鉴权。
    backend = ContextHubBackend(HUB_IDENTIFIER)

    agent = create_deep_agent(model=llm, backend=backend)

    result = agent.invoke(
        {"messages": [{"role": "user", "content": "当前目录有哪些文件"}]},
        config={"recursion_limit": 50},
    )
    print(result["messages"][-1].content)
    print(f"\n写入的文件可以在 LangSmith 网页上打开仓库 {HUB_IDENTIFIER} 查看（含 Git 提交历史）。")


print("=" * 62)
print("DeepAgents 后端⑤：ContextHubBackend（LangSmith Hub 云端仓库）")
print("=" * 62)

if not langsmith_api_key:
    # ---------- 降级路径：不抛异常，只给清晰的中文指引 ----------
    print(
        "\n[跳过] 未检测到 LANGSMITH_API_KEY 环境变量，本示例不连接 LangSmith。\n"
        "\n为什么跳过：\n"
        "  ContextHubBackend 把文件写进 LangSmith Hub 的 agent 仓库，\n"
        "  构造后端时就会 new 一个 langsmith.Client() 去鉴权；\n"
        "  没有密钥会直接抛凭据异常 —— 这是外部账号依赖，不是代码问题。\n"
        "\n想真正运行，请依次完成：\n"
        "  1. 注册 https://smith.langchain.com 并创建 API Key；\n"
        "  2. 在本机设置环境变量（PowerShell，当前会话生效）：\n"
        '         $env:LANGSMITH_API_KEY = "lsv2_pt_xxxxxxxx"\n'
        "     （永久生效请用 setx，改完重开终端）\n"
        f"  3. 把本文件里的 HUB_IDENTIFIER 从 {HUB_IDENTIFIER!r} 改成你自己的仓库，\n"
        '     格式为 "组织名/仓库名"，也可以是 "-/仓库名"（个人空间）；\n'
        "  4. 重新运行本文件。\n"
    )

    print(
        "不依赖 LangSmith 的等效做法（本项目 04_后端_Store_jxsd.py 用的就是它）：\n"
        "  用 StoreBackend + 固定 namespace 当共享枢纽，\n"
        "  同样能让多个会话/线程共享同一份文件，只是少了 Git 版本历史。\n"
    )
else:
    # ---------- 正常路径 ----------
    print(f"\n已检测到 LANGSMITH_API_KEY（长度 {len(langsmith_api_key)}），连接 Hub 仓库 {HUB_IDENTIFIER} …\n")
    try:
        run_with_context_hub()
    except Exception as exc:                      # noqa: BLE001 —— 外部服务异常要转成中文提示
        print(f"\n[失败] 连接 LangSmith Hub 出错：{type(exc).__name__}: {exc}")
        print("请检查：API Key 是否有效、HUB_IDENTIFIER 仓库是否存在、网络是否能访问 smith.langchain.com。")

### 预期输出

```text
==============================================================
DeepAgents 后端⑤：ContextHubBackend（LangSmith Hub 云端仓库）
==============================================================

[跳过] 未检测到 LANGSMITH_API_KEY 环境变量，本示例不连接 LangSmith。

为什么跳过：
  ContextHubBackend 把文件写进 LangSmith Hub 的 agent 仓库，
  构造后端时就会 new 一个 langsmith.Client() 去鉴权；
  没有密钥会直接抛凭据异常 —— 这是外部账号依赖，不是代码问题。

想真正运行，请依次完成：
  1. 注册 https://smith.langchain.com 并创建 API Key；
  2. 在本机设置环境变量（PowerShell，当前会话生效）：
         $env:LANGSMITH_API_KEY = "lsv2_pt_xxxxxxxx"
     （永久生效请用 setx，改完重开终端）
  3. 把本文件里的 HUB_IDENTIFIER 从 'your-org/my-agent' 改成你自己的仓库，
     格式为 "组织名/仓库名"，也可以是 "-/仓库名"（个人空间）；
  4. 重新运行本文件。

不依赖 LangSmith 的等效做法（本项目 04_后端_Store_jxsd.py 用的就是它）：
  用 StoreBackend + 固定 namespace 当共享枢纽，
  同样能让多个会话/线程共享同一份文件，只是少了 Git 版本历史。
```

> ⚠️ 上面这段是本机「未配置 LANGSMITH_API_KEY」时**固定不变**的降级文案；若你配了密钥，
> 会走「已检测到…」分支并真正连接 LangSmith，输出会完全不同（由模型决定，每次不同）。

### 5.4 什么时候用它

- **选它**：已经在用 LangSmith 的团队，想让 Agent 产出像代码一样可回溯、可 diff、可回滚。
- **换人**：不能联网 / 不想依赖外部账号 → StoreBackend（第 2 节）。

## 6. Sandbox 后端（Sandbox）

### 6.1 它把状态存哪

LocalShellBackend 把宿主机终端交出去，生产环境不敢这么干。Sandbox 的思路是：
**文件操作 API 完全不变，但 execute 落到隔离环境里**。DeepAgents 定义了统一的沙箱接口
（`SandboxBackendProtocol`），换 Provider 就像换数据库驱动：

| Provider | 位置 | 说明 | 需额外安装 / 配置 |
|---|---|---|---|
| LangSmithSandbox | 云端 | LangChain 官方云端沙箱（需付费开通） | LANGSMITH_API_KEY |
| E2B | 云端 | 云沙箱（类似远程 Docker） | pip install e2b + API Key |
| Daytona | 云端 | 开发环境管理平台 | pip install daytona + API Key |
| Modal | 云端 | Modal 云函数平台 | pip install modal + Token |
| Runloop | 云端 | Runloop 沙箱平台 | Runloop API Key |
| 本地 VFS | 本地 | = LocalShellBackend，可直接用，不隔离进程 | 内置，无需额外安装 |

所以「本地 VFS」那一行就是第 4 节的 LocalShellBackend —— 它是最省事的沙箱，但**不隔离进程**。

### 6.2 课案原版：只做说明（stub）

课案原版（08_后端_Sandbox.py）因为沙箱需要云端/本地服务端支撑，**只做导入与说明，不实际执行**。

In [ ]:
# ---------- 课案原版（08_后端_Sandbox.py） ----------
# LangSmithSandbox 需要一个已创建的 LangSmith 沙箱实例：
#   from langsmith.sandbox import Sandbox
#   sandbox = Sandbox.create(...)  # 需配置 LANGSMITH_API_KEY
#   backend = LangSmithSandbox(sandbox)
# 未配置 LangSmith 时会直接抛出缺少凭据的异常——这是预期的。

# Opensandbox 版本（需安装 deepagents-opensandbox 并启动服务端）：
#   uv add deepagents-opensandbox opensandbox opensandbox-server
#   （在 WSL/Linux 中启动服务端）opensandbox-server
#   from deepagents_opensandbox import OpensandboxBackend
#   backend = OpensandboxBackend(server_url="http://localhost:8000")

# 由于沙箱需要云端/本地服务端支撑，本文件仅做导入与说明，不实际执行。
print("沙箱后端说明输出完毕（真实执行需要 LangSmith 或 Opensandbox 服务端）")

### 预期输出

```text
沙箱后端说明输出完毕（真实执行需要 LangSmith 或 Opensandbox 服务端）
```

### 6.3 完整版：两套代码 + 前置体检（本机走降级路径）

完整版（08_后端_Sandbox_jxsd.py）包含课案里的两套代码（LangSmithSandbox 云端付费 / opensandbox 自建 Docker），
都封装成函数**不调用**；`__main__` 只做**前置条件体检 + 中文说明**。

> 为什么「此代码不用跑，收费」：LangSmithSandbox 落在官方云端，`client.sandbox(...)` 一执行就开始计费，
> 直到 `sandbox.delete()` 才停。所以课案直接把那 16 行标成「不用跑」，本文件也把它封进函数不调用。

In [ ]:
# ---------- 完整版（08_后端_Sandbox_jxsd.py） ----------
# LangSmith / opensandbox 的凭据都不在 config.py 里，按规范直接读环境变量
LANGSMITH_API_KEY = os.environ.get("LANGSMITH_API_KEY", "")


# ============================================================
# 课案代码 ①：LangSmithSandbox —— 先创建远程沙箱，再包进 backend（需 LANGSMITH_API_KEY）
# ============================================================
# 此代码不用跑，收费
def demo_langsmith_sandbox() -> None:
    """课案 ① 的可执行版本（本文件不会调用）。

    运行前提：
        - 已付费开通 LangSmith Sandbox
        - 环境变量 LANGSMITH_API_KEY 已设置
        - 云端已存在名为 my-sandbox 的沙箱
    """
    try:
        from langsmith.sandbox import SandboxClient
    except ImportError as exc:                       # pragma: no cover —— 缺包时给中文提示
        print(f"[跳过] 未能导入 langsmith.sandbox：{exc}")
        return

    # 先创建远程沙箱，再包进 backend
    # ⚠️ 注意顺序：这一行一执行，云端就开始计费（详见文件头第四节）
    client = SandboxClient()
    sandbox = client.sandbox(name="my-sandbox")
    try:
        # 把远程沙箱包成 backend —— 之后 Agent 的文件工具与 execute 都落到这台沙箱里
        agent = create_deep_agent(model=llm, backend=LangSmithSandbox(sandbox))
        result = agent.invoke(
            {"messages": [{"role": "user", "content": "当前目录有哪些文件"}]},
            config={"recursion_limit": 50},
        )
        # 沙箱里执行命令同样会走多轮（ls → 读 → 回答），所以也放宽步数上限
        print(result["messages"][-1].content)
    finally:
        # 沙箱是云端计费资源，用完必须销毁
        try:
            sandbox.delete()
            print("已销毁远程沙箱。")
        except Exception as exc:                     # noqa: BLE001
            print(f"[警告] 沙箱销毁失败，请到 LangSmith 控制台手动清理：{exc}")


# ============================================================
# 课案代码 ②：本地可跑的沙箱（opensandbox + Docker）
# ============================================================
# 安装依赖包： pip install deepagents-opensandbox deepagents opensandbox opensandbox-server
# 生成默认配置： opensandbox-server init-config ~/.sandbox.toml --example docker
# 启动沙箱服务，选输入YES：opensandbox-server
def demo_opensandbox() -> None:
    """课案 ② 的可执行版本（本文件不会调用）。

    第 4、5 步之间对应课案原文的流程；第 6 步 sandbox.kill() 是收尾清理，
    对应课案标题「6. 销毁沙箱，清理本地 Docker 资源」。
    """
    try:
        from datetime import timedelta

        from opensandbox.sync.sandbox import SandboxSync
        from deepagents_opensandbox import OpensandboxBackend
    except ImportError as exc:                       # pragma: no cover —— 缺包时给中文提示
        print(f"[跳过] 本地沙箱依赖未安装：{exc}")
        print("       安装命令：pip install deepagents-opensandbox opensandbox opensandbox-server")
        return

    # 1. 配置本地沙箱连接（无需 API Key，直接连接本机 8080 服务）
    os.environ["OPEN_SANDBOX_DOMAIN"] = "http://localhost:8080"

    # 2. 在本地 Docker 中创建隔离沙箱容器
    sandbox = SandboxSync.create(
        image="python:3.12-slim",        # 基础 Docker 镜像，可自定义
        timeout=timedelta(seconds=300),  # 沙箱总存活时长
    )
    try:
        execution = sandbox.commands.run("python --version")
        print(execution.logs.stdout[0].text)

        # 3. 包装为 DeepAgents 兼容的执行后端
        backend = OpensandboxBackend(sandbox=sandbox)

        # 4. 创建 DeepAgent，绑定沙箱后端
        agent = create_deep_agent(
            model=llm,
            system_prompt="你是具备隔离沙箱执行权限的编程助手，所有代码和命令都在沙箱中安全运行。",
            backend=backend,
        )

        # 5. 执行任务
        result = agent.invoke(
            {"messages": [{"role": "user", "content": "创建一个 hello.py 并运行它"}]},
            config={"recursion_limit": 50},
        )
        for message in result["messages"]:
            print(message.content)
    finally:
        # 6. 销毁沙箱，清理本地 Docker 资源
        sandbox.kill()
        print("已销毁沙箱，本地 Docker 资源已清理。")


# ============================================================
# 降级路径：体检 + 中文说明（本文件实际执行的只有这一段）
# ============================================================
print("=" * 68)
print("DeepAgents 后端⑥：Sandbox（隔离容器执行）")
print("=" * 68)
print(
    "\n本文件按课案要求**不做真实调用**，只做说明与前置条件体检。\n"
    "原因见下：\n"
)

# ---------- 体检 ①：云端沙箱 ----------
print("① LangSmithSandbox（云端沙箱）")
print("   课案原话：# 此代码不用跑，收费")
if LANGSMITH_API_KEY:
    # 只打印长度，绝不回显密钥本身
    print(f"   环境变量 LANGSMITH_API_KEY：已设置（长度 {len(LANGSMITH_API_KEY)}）")
else:
    print("   环境变量 LANGSMITH_API_KEY：**未设置**")
print("   还需：LangSmith Sandbox 已付费开通 + 云端已存在沙箱实例。")
print("   ⚠️ 本文件不会自动连接 LangSmith；确认要跑请自行调用 demo_langsmith_sandbox()。\n")

# ---------- 体检 ②：本地 opensandbox（Docker） ----------
print("② opensandbox（本地 Docker 沙箱）")
missing = []
for module_name in ("opensandbox", "deepagents_opensandbox"):
    try:
        # __import__ 而不是 import：模块名是变量，且这里只想探测「在不在」
        __import__(module_name)
        print(f"   {module_name:<24} 已安装")
    except ImportError:
        missing.append(module_name)
        print(f"   {module_name:<24} **未安装**")

docker_ok = False
try:
    # 只查 docker 可执行文件在不在 PATH 里，不启动任何容器
    import shutil
    docker_ok = shutil.which("docker") is not None
except Exception:                                # noqa: BLE001
    # 探测本身失败也不该让体检崩掉 —— 一律当作「不可用」继续往下走
    docker_ok = False
print(f"   docker 可执行文件{'已找到' if docker_ok else '**未找到**'}")

if missing or not docker_ok:
    print(
        "\n   [跳过] 本地沙箱的前置条件不满足，无法演示。\n"
        "   准备步骤（课案原文，建议在 Linux / WSL 里做）：\n"
        "       pip install deepagents-opensandbox deepagents opensandbox opensandbox-server\n"
        "       opensandbox-server init-config ~/.sandbox.toml --example docker\n"
        "       opensandbox-server        # 启动沙箱服务，交互提示选 YES\n"
        "   然后确认本机 8080 端口可用（OPEN_SANDBOX_DOMAIN=http://localhost:8080）。\n"
    )

# ---------- 结论与替代方案 ----------
print("=" * 68)
print(
    "结论：两个沙箱方案都依赖外部条件（付费云端 / Docker + 额外 pip 包），\n"
    "      本项目规范禁止新增依赖，故本文件只打印说明并正常退出。\n"
    "\n不花钱也能体验「Agent 执行命令」的两条路：\n"
    "  - 06_后端_LocalShell_jxsd.py：内置的「本地 VFS」，直接能跑，但不隔离进程；\n"
    "  - 09_后端_Composite_jxsd.py：把 execute 交给沙箱、文件路由到磁盘的混合方案。\n"
    "\n真要上生产，请用 Sandbox 类后端（LangSmith / E2B / Daytona / Modal / Runloop\n"
    "或自建 Docker），并配合 10_人工审核_jxsd.py 做危险操作审批。"
)

### 预期输出

```text
====================================================================
DeepAgents 后端⑥：Sandbox（隔离容器执行）
====================================================================

本文件按课案要求不做真实调用，只做说明与前置条件体检。
原因见下：

① LangSmithSandbox（云端沙箱）
   课案原话：# 此代码不用跑，收费
   环境变量 LANGSMITH_API_KEY：未设置
   还需：LangSmith Sandbox 已付费开通 + 云端已存在沙箱实例。
   ⚠️ 本文件不会自动连接 LangSmith；确认要跑请自行调用 demo_langsmith_sandbox()。

② opensandbox（本地 Docker 沙箱）
   opensandbox               未安装
   deepagents_opensandbox    未安装
   docker 可执行文件（已找到 / 未找到）

   [跳过] 本地沙箱的前置条件不满足，无法演示。
   ...

====================================================================
结论：两个沙箱方案都依赖外部条件（付费云端 / Docker + 额外 pip 包），
      本项目规范禁止新增依赖，故本文件只打印说明并正常退出。
...
```

> ⚠️ 本格输出**因机器而异**：`opensandbox` / `deepagents_opensandbox` 是否安装、
> `docker 可执行文件` 是否在 PATH 里，取决于本机环境。上面是「未安装 / 未找到」时的形态。别逐字比对。

### 6.4 什么时候用它

- 要跑代码但**不在乎隔离** → LocalShellBackend（第 4 节）；
- 要跑代码且**必须隔离** → 本节，代价是外部依赖（付费云端或自建 Docker）；
- 连代码都不用跑、只要改文件 → FilesystemBackend（第 3 节）就够，别上沙箱。

## 7. Composite 后端（CompositeBackend）

### 7.1 它把状态存哪

前面六个后端各管一摊，但真实项目里一个 Agent 往往**同时**需要临时草稿、长期记忆、真实项目文件。
CompositeBackend 自己**通常不负责真正存储**，而是按路径前缀把操作转发给对应后端 —— 后端层的「路由器」。

路由语义（关键）：**先按前缀匹配、匹配上就把前缀剥掉再转给目标后端**；前缀必须**结尾带斜杠**才按目录匹配；
没匹配上的走 `default`。

> **建议**：大多数场景用 CompositeBackend。因为 DeepAgents 内部会自动向后端写入工具结果
> （`/large_tool_results/`）和对话历史（`/conversation_history/`），所以 `default` 最好用 `StateBackend`
> （内存、自动清退），让这些内部文件不落盘；`/workspace/` 路由到 FilesystemBackend 操作真实项目文件；
> `/memories/` 路由到 StoreBackend 做长期记忆。

### 7.2 课案原版：/final/ 走磁盘

课案原版最简路由：`/final/**` 走磁盘，其他走 State。

In [ ]:
# ---------- 课案原版（09_后端_Composite.py） ----------
# 原版写 Path("tmp_deepagents_composite")；notebook 里改到专属子目录，避免污染仓库根
out_dir = NB_DIR / "tmp_nb_work" / "backend_composite_原版"
out_dir.mkdir(exist_ok=True)

# 路由规则：/final/** 走磁盘，其他走 State
backend = CompositeBackend(
    default=StateBackend(),
    routes={"/final/": FilesystemBackend(root_dir=str(out_dir))},
)

agent = create_deep_agent(
    model=llm,
    backend=backend,
    system_prompt="草稿写临时文件，最终成果写入 /final/ 目录。",
)

result = agent.invoke(
    {"messages": [("user", "把最终报告写到 /final/report.md，内容是：项目验收通过")]},
    config={"recursion_limit": 50},
)
print("AI：", result["messages"][-1].content)
print("磁盘文件：", [p.name for p in out_dir.iterdir()])

### 预期输出

```text
AI：（模型回复，措辞每次不同）
磁盘文件： ['report.md']
```

> ⚠️ 本格输出含模型生成内容（`AI：` 后正文）。`磁盘文件：` 后应看到模型写入 `/final/` 的
> `report.md`（文件名可能因模型而异）。别逐字比对。

### 7.3 完整版：三路路由实测

完整版（09_后端_Composite_jxsd.py）实现课案的组合示例：`/workspace/` → 磁盘、
`/memories/` → Store、其他 → State，并用课后核对把三份数据分别去了哪**验证**出来。

In [ ]:
# ---------- 完整版（09_后端_Composite_jxsd.py） ----------
# /workspace/ 这条路由最终落到这个真实目录（原版是 __file__ 同级目录，notebook 里改到专属子目录）
WORKSPACE_DIR = NB_DIR / "tmp_nb_work" / "backend_composite"

# 本地演示用内存 Store；生产换成 PostgresStore.from_conn_string(settings.pg_uri)
store = InMemoryStore()

agent = create_deep_agent(
    model=llm,
    backend=CompositeBackend(
        default=StateBackend(),          # 兜底：/large_tool_results/ 等内部文件留在内存
        routes={
            # 前缀要用**结尾带斜杠**的写法，才会按「目录」匹配
            "/workspace/": FilesystemBackend(root_dir=str(WORKSPACE_DIR), virtual_mode=True),
            "/memories/": StoreBackend(namespace=lambda rt: ("memories", "1")),
        },
    ),
    # 本地要自己把 store 交给图（同 04 文件里的解释：上平台后由平台自动注入）
    store=store,
    system_prompt=(
        "你是项目助手，路径规则如下："
        "真实项目文件写到 /workspace/ 下；需要长期记住的信息写到 /memories/ 下；"
        "其他临时草稿随便放。"
    ),
)

WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

# ---------- ① 课案任务：往 /workspace/ 写文件 → 应该落到真实磁盘 ----------
print("===== ① 在 workspace 里创建一个 txt 文件 =====")
result = agent.invoke(
    {"messages": [{"role": "user", "content": "在workspace里面创建一个txt文件"}]},
    config={"recursion_limit": 50},
)
print(result["messages"][-1].content)

# ---------- ② 课案任务：往 /memories/ 写文件 → 应该进 Store ----------
print("\n===== ② 在 memories 里记一条长期信息 =====")
result = agent.invoke(
    {"messages": [{"role": "user", "content": "把「用户偏好：回答要简短」写进 /memories/prefs.md"}]},
    config={"recursion_limit": 50},
)
print(result["messages"][-1].content)

# ---------- ③ 验证路由：三份数据分别去了哪 ----------
print("\n===== ③ 路由结果验证 =====")

print("[真实磁盘] /workspace/ →", WORKSPACE_DIR)
disk_files = sorted(p.relative_to(WORKSPACE_DIR) for p in WORKSPACE_DIR.rglob("*") if p.is_file())
if disk_files:
    for rel in disk_files:
        print(f"    ✅ {rel}")
else:
    print("    （本轮没有文件落到磁盘）")

print("[LangGraph Store] /memories/ → namespace ('memories', '1')")
items = store.search(("memories", "1"))
if items:
    for item in items:
        print(f"    ✅ {item.key}")
else:
    print("    （本轮没有文件写进 Store）")

# StateBackend 兜底：这些文件只存在于本次运行的 state 里，函数返回即丢
state_files = sorted((result.get("files") or {}).keys())
print("[StateBackend 兜底] 其他路径 → 内存")
if state_files:
    for path in state_files:
        print(f"    ✅ {path}")
else:
    print("    （本轮 state 里没有额外文件）")

print(
    "\n结论：同一个 Agent、同一套文件工具，靠路径前缀把数据分流到了三个地方。\n"
    "      这就是课案推荐「大多数场景用 CompositeBackend」的原因。"
)

### 预期输出

```text
===== ① 在 workspace 里创建一个 txt 文件 =====
（模型回复，措辞每次不同）

===== ② 在 memories 里记一条长期信息 =====
（模型回复，措辞每次不同）

===== ③ 路由结果验证 =====
[真实磁盘] /workspace/ → F:\ProGram\Python_Base\Agent\03_deepagents\tmp_nb_work\backend_composite
    ✅ xxx.txt
[LangGraph Store] /memories/ → namespace ('memories', '1')
    ✅ /prefs.md
[StateBackend 兜底] 其他路径 → 内存
    （本轮 state 里没有额外文件）

结论：同一个 Agent、同一套文件工具，靠路径前缀把数据分流到了三个地方。
      这就是课案推荐「大多数场景用 CompositeBackend」的原因。
```

> ⚠️ 本格输出含模型生成内容与**本机路径**：模型回复、落盘文件名、Store 里的 key **每次不同**。
> 稳定的只有各段标题、`[真实磁盘]` / `[LangGraph Store]` / `[StateBackend 兜底]` 三段结构和结论文案。
> 若模型某轮没写文件，对应段会打印「（本轮没有文件落到磁盘 / 写进 Store / state 里没有额外文件）」。别逐字比对。

### 7.4 什么时候用它

三句话收口（把 7 个后端压成决策）：

1. 只改文件、不跑代码 → **FilesystemBackend**；
2. 要跑代码、不在乎隔离 → **LocalShellBackend**（本地开发）；
3. 要跑代码、必须隔离 → **Sandbox**（付费云端 or 自建 Docker）。

其余 State / Store / ContextHub 管「文件存哪」，与「能不能执行」正交，可按持久化需求自由组合——
组合的方式就是 CompositeBackend。

## 小结

| 后端 | 物理存储 | 资源管理器能看到吗 | 生命周期 |
|---|---|---|---|
| StateBackend | LangGraph state（Python dict） | ❌ | 同 thread，进程死即丢 |
| FilesystemBackend | 你的真实磁盘 | ✅ | 永久 |
| LocalShellBackend | 你的真实磁盘 | ✅ | 永久 |
| StoreBackend | LangGraph Store（内存 / PostgreSQL） | ❌ | 跨 thread，取决于 Store |
| ContextHubBackend | LangSmith Hub 云端仓库（Git 版本管理） | ❌（网页可看） | 永久 + 版本历史 |
| Sandbox | 沙箱容器内临时文件系统 | ❌ | 沙箱销毁即丢 |
| CompositeBackend | 取决于路由规则（以上任意组合） | — | — |

- 「能不能跑代码」和「文件存哪」是**两个正交的问题**：State / Store / ContextHub / Composite 管存哪，
  LocalShell / Sandbox 额外管「能不能 execute」。
- **默认是 StateBackend**，草稿纸定位；要长期记住 → Store；要版本历史 → ContextHub；
  要碰真实磁盘 → Filesystem；要跑代码 → LocalShell（不隔离）/ Sandbox（隔离）；
  都要 → Composite。

## 常见坑

1. **`virtual_mode=True` 管不住 shell**：它只拦文件工具里的 `../` 和绝对路径；
  LocalShellBackend 一旦开了 execute，命令照样能访问任意路径，路径拦截形同虚设。
2. **非沙箱后端也会看到 `execute` 工具**：deepagents 统一挂了 execute，只在沙箱类后端上真正生效；
  判断「能不能执行」要看后端类型，别只看工具列表。
3. **StoreBackend 的 `namespace` 是隔离边界**：写错（比如漏了 user_id）会导致不同用户共享同一个抽屉。
4. **CompositeBackend 前缀要结尾带斜杠**：`/workspace/` 才会按目录匹配，`/workspace` 不剥前缀、行为不同。
5. **StateBackend 必须配 checkpointer**：文件随 checkpoint 走，不传 checkpointer 第二轮 invoke 就是全新的。
6. **LocalShell 在 Windows 上是 cmd.exe 不是 bash**：模型容易敲 `ls` / `pwd` / `cat`，
  在 cmd.exe 里必然失败，必须把「当前 shell 是什么」写进 system_prompt，否则会撞 `GraphRecursionError`。
7. **LangSmithSandbox 一建就计费**：`client.sandbox(...)` 从执行那刻起计费，用完必须 `sandbox.delete()`。

## 官方链接

- DeepAgents Backends（本课唯一官方出处）：<https://docs.langchain.com/oss/python/deepagents/backends>
- DeepAgents 总览：<https://docs.langchain.com/oss/python/deepagents/overview>